# Construct quadratic program

In [1]:
import numpy as np
import networkx as nx

from scipy.optimize import minimize

from qiskit.primitives import (
    BackendSamplerV2,
    BackendEstimator,
    BackendEstimatorV2,
    StatevectorEstimator,
    Estimator,
)
from qiskit.circuit.library import EfficientSU2, TwoLocal
from qiskit_optimization.applications import Maxcut
from qiskit_optimization.algorithms import MinimumEigenOptimizer
from qiskit_algorithms import NumPyMinimumEigensolver, VQE
from qiskit_algorithms.optimizers import NELDER_MEAD, COBYLA
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import (
    Estimator as AerEstimator,
    EstimatorV2 as AerEstimatorV2,
)

from lcc import LCCVQE

In [14]:
n = 10
G = nx.random_regular_graph(3, n, seed=0)
max_cut = Maxcut(G)
qp = max_cut.to_quadratic_program()
print(qp.prettyprint())

Problem name: Max-cut

Maximize
  -2*x_0*x_3 - 2*x_0*x_5 - 2*x_0*x_7 - 2*x_1*x_2 - 2*x_1*x_4 - 2*x_1*x_6
  - 2*x_2*x_3 - 2*x_2*x_8 - 2*x_3*x_5 - 2*x_4*x_6 - 2*x_4*x_9 - 2*x_5*x_8
  - 2*x_6*x_9 - 2*x_7*x_8 - 2*x_7*x_9 + 3*x_0 + 3*x_1 + 3*x_2 + 3*x_3 + 3*x_4
  + 3*x_5 + 3*x_6 + 3*x_7 + 3*x_8 + 3*x_9

Subject to
  No constraints

  Binary variables (10)
    x_0 x_1 x_2 x_3 x_4 x_5 x_6 x_7 x_8 x_9



## Exact solution

In [20]:
exact = MinimumEigenOptimizer(NumPyMinimumEigensolver())
result_max_cut = exact.solve(qp)
print(result_max_cut.prettyprint())

objective function value: 13.0
variable values: x_0=0.0, x_1=0.0, x_2=1.0, x_3=0.0, x_4=1.0, x_5=1.0, x_6=1.0, x_7=1.0, x_8=0.0, x_9=0.0
status: SUCCESS


# Solve using LCC

In [28]:
estimator = StatevectorEstimator()
# estimator = None
optimizer = COBYLA()

solver = LCCVQE(
    qp,
    estimator=estimator,
    optimizer=optimizer,
)

initial_params = np.random.rand(2 * qp.get_num_vars()) * 2 * np.pi
print('Initial params:', initial_params.view())
solver.compute_energy(initial_params)

Initial params: [4.07108741 4.44188006 3.1664095  1.15398186 4.38463917 4.86515642
 0.7034524  0.27366768 2.77492114 4.20729134 4.3630435  0.8865588
 4.5910803  5.19604462 3.55903953 2.99959926 1.61500756 2.90281145
 0.89864938 5.20864724]


0.408447265625

In [16]:
# ansatz = EfficientSU2(
#     num_qubits=qp_max_cut.get_num_vars(),
#     su2_gates='ry',
#     entanglement='circular',
#     reps=1
# )
ansatz = TwoLocal(
    num_qubits=qp.get_num_vars(),
    rotation_blocks='ry',
    entanglement_blocks='cz',
    entanglement='circular',
    reps=1,
)
result = estimator.run([(ansatz, solver.hamiltonian, initial_params)]).result()[0]
result.data.evs

array(0.62592011)

In [ ]:
# estimator = Estimator()
# optimizer = COBYLA()

# solver = LCCVQE(
#     qp_max_cut,
#     estimator=estimator,
#     optimizer=optimizer,
# )

# initial_params = np.random.rand(2 * qp_max_cut.get_num_vars()) * 2 * np.pi
# print('Initial params:', initial_params.view())
# solver.compute_energy_v1(initial_params)

In [29]:
result = solver.solve(initial_params)
print(result)

{   'fun': -5.43505859375,
    'jac': None,
    'nfev': 270,
    'nit': None,
    'njev': None,
    'x': array([ 6.48105227,  6.34933368,  5.61706706,  0.16727704,  6.12105658,
        6.04047848, -0.07256694,  0.40981583,  3.09739492,  5.61415042,
        6.04717363, -0.0175762 ,  3.84852793,  6.1185348 ,  3.58398114,
        3.38063133,  3.05834113,  3.49915217,  3.17241038,  5.6311517 ])}


In [ ]:
# def obj_func(params):
#     return solver.compute_energy(params)

# result = minimize(obj_func, initial_params, method='cobyla')
# result

In [26]:
print('Expectation:', -(result.fun + solver.offset))

Expectation: 12.999999960222304


In [44]:
solver.offset

-7.5

# Solve without LCC

In [ ]:
ham, offset = qp.to_ising()

backend = AerSimulator(method='statevector')
estimator = Estimator()
ansatz = EfficientSU2(
    num_qubits=qp.get_num_vars(),
    su2_gates='ry',
    entanglement='circular',
    reps=1
)
vqe = VQE(estimator, ansatz, optimizer, initial_point=initial_params)
result = vqe.compute_minimum_eigenvalue(ham)
print(result)

In [111]:
result = estimator.run([(ansatz, ham, initial_params)]).result()[0]
result.data.evs

array(-0.36011752)

# Reconstructing the quasi-prob distribution

## Brute force (run the full circuit)

In [ ]:
backend = AerSimulator(method='matrix_product_state')
sampler = BackendSamplerV2(backend=backend)
ansatz = TwoLocal(
    num_qubits=qp.get_num_vars(),
    rotation_blocks='ry',
    entanglement_blocks='cz',
    entanglement='circular',
    reps=1,
)

result = sampler.run([ansatz, solver.optimal_params]).result()